In [ ]:
# @title ランバト管理ツール
#@markdown PC表示のがいいｶﾓ
import ipywidgets as widgets
from IPython.display import display, clear_output
from google.colab import drive
from google.colab import files
import json
import os
import re
import requests
import io

DRIVE_MOUNT_POINT = '/content/drive'
FILE_NAME = 'tuyoiuser.json'
OLD_FILE_NAME = 'jirai_user_list_v2.json'
SAVE_DIR = os.path.join(DRIVE_MOUNT_POINT, 'MyDrive')
FILE_PATH = os.path.join(SAVE_DIR, FILE_NAME)
OLD_FILE_PATH = os.path.join(SAVE_DIR, OLD_FILE_NAME)

if not os.path.exists(DRIVE_MOUNT_POINT):
    drive.mount(DRIVE_MOUNT_POINT)

class UserManager:
    def __init__(self, filepath, old_filepath):
        self.filepath = filepath
        self.data = self.load_data(old_filepath)

    def load_data(self, old_filepath):
        if os.path.exists(self.filepath):
            try:
                with open(self.filepath, 'r', encoding='utf-8') as f:
                    return json.load(f)
            except: pass
        if os.path.exists(old_filepath):
            try:
                with open(old_filepath, 'r', encoding='utf-8') as f:
                    c = json.load(f)
                    return {n: 1 for n in c} if isinstance(c, list) else c
            except: pass
        return {}

    def save_data(self):
        try:
            with open(self.filepath, 'w', encoding='utf-8') as f:
                json.dump(self.data, f, ensure_ascii=False, indent=4)
        except: pass

    def sanitize(self, text):
        if not isinstance(text, str): return ""
        return re.sub(r'[^a-zA-Zぁ-んァ-ンー]', '', text)

    def add_user(self, raw_name):
        name = self.sanitize(raw_name)
        if not name: return False, "文字を入力してください", "", 0
        if name in self.data:
            self.data[name] += 1
            msg = f"⚠ <b>{name}</b> (Lv.{self.data[name]})"
        else:
            self.data[name] = 1
            msg = f"🆕 <b>{name}</b> (Lv.1)"
        self.save_data()
        return True, msg, name, self.data[name]

    def check_safety(self, names_list):
        safe = []
        mines = []
        for raw in names_list:
            clean = self.sanitize(raw)
            if not clean: continue
            if clean in self.data:
                mines.append((clean, self.data[clean]))
            else:
                safe.append(clean)
        mines.sort(key=lambda x: x[1], reverse=True)
        return safe, mines

    def delete_user(self, name):
        if name in self.data:
            del self.data[name]
            self.save_data()
            return True
        return False

    def search_in_list(self, query):
        clean_q = self.sanitize(query)
        if not clean_q and query.strip() != "": return []
        matches = []
        for name, lv in self.data.items():
            if clean_q == "" or clean_q in name:
                matches.append((name, lv))
        return sorted(matches, key=lambda x: x[1], reverse=True)

    def merge_data(self, content):
        added, updated = 0, 0
        src = {n: 1 for n in content} if isinstance(content, list) else content
        for n, c in src.items():
            cn = self.sanitize(n)
            if not cn: continue
            if not isinstance(c, int): c = 1
            if cn in self.data:
                self.data[cn] += c
                updated += 1
            else:
                self.data[cn] = c
                added += 1
        self.save_data()
        return added, updated, "OK"

    def send_to_webhook(self, url):
        if not url.startswith("http"): return False, "URLエラー"
        try:
            self.save_data()
            with open(self.filepath, 'rb') as f:
                payload = {'file': (FILE_NAME, f, 'application/json')}
                requests.post(url, files=payload)
            return True, "送信成功"
        except Exception as e:
            return False, str(e)

manager = UserManager(FILE_PATH, OLD_FILE_PATH)

css = widgets.HTML("""
<style>
    .app-container {
        background-color: #1a1a1d !important;
        color: #e0e0e0 !important;
        font-family: 'Helvetica Neue', Arial, sans-serif;
        padding: 0;
        border-radius: 12px;
        width: 850px;
        margin: 20px auto;
        box-shadow: 0 10px 40px rgba(0,0,0,0.7);
        border: 1px solid #333;
        overflow: hidden;
    }
    .app-header { background-color: #212125; padding: 20px; border-bottom: 1px solid #333; }
    .app-title {
        font-size: 22px; font-weight: 800; color: #fff;
        border-left: 5px solid #00e676; padding-left: 15px; letter-spacing: 1px;
    }
    .tab-bar { display: flex; background-color: #121214; }
    .custom-tab-btn {
        flex: 1; background: transparent !important; color: #888 !important;
        border: none !important; border-bottom: 3px solid transparent !important;
        height: 50px !important; font-weight: bold !important; font-size: 14px !important;
        margin: 0 !important; border-radius: 0 !important;
    }
    .active-tab {
        background: #212125 !important; color: #fff !important;
        border-bottom: 3px solid #00e676 !important;
    }
    .content-area { padding: 30px; background-color: #212125; min-height: 450px; box-sizing: border-box; }

    .widget-text {
        width: 96% !important;
        margin: 0 auto 10px auto !important;
    }
    .widget-text input[type="text"] {
        width: 100% !important;
        box-sizing: border-box !important;
        background-color: #161618 !important;
        color: #fff !important;
        border: 1px solid #444 !important;
        border-radius: 4px !important;
        padding: 12px !important;
        font-size: 14px !important;
        margin: 0 !important;
    }
    .widget-text input:focus { border-color: #00e676 !important; outline: none !important; }

    .hero-safe {
        background: linear-gradient(135deg, rgba(0, 230, 118, 0.1) 0%, rgba(0, 230, 118, 0.05) 100%);
        border: 2px solid #00e676; color: #00e676;
        padding: 20px; border-radius: 8px; margin-bottom: 20px;
        text-align: center; box-shadow: 0 0 15px rgba(0, 230, 118, 0.2);
    }
    .hero-warning {
        background: linear-gradient(135deg, rgba(255, 145, 0, 0.1) 0%, rgba(255, 145, 0, 0.05) 100%);
        border: 2px solid #ff9100; color: #ff9100;
        padding: 20px; border-radius: 8px; margin-bottom: 20px;
        text-align: center; box-shadow: 0 0 15px rgba(255, 145, 0, 0.2);
    }

    .list-mine {
        background: rgba(255, 23, 68, 0.1); border-left: 4px solid #ff1744;
        color: #ff1744; padding: 10px 15px; margin-bottom: 8px; border-radius: 4px;
        display: flex; justify-content: space-between; align-items: center;
    }

    .widget-upload { width: 96% !important; margin: 0 auto !important; }
    .widget-upload > .widget-upload-button {
        width: 100% !important; background-color: #2b2b30 !important;
        color: #fff !important; border: 1px dashed #666 !important;
        font-weight: bold !important; padding: 15px !important; height: auto !important;
        box-sizing: border-box !important;
    }
</style>
""")

layout_full = widgets.Layout(width='95%')
layout_btn = widgets.Layout(width='95%', height='50px')

v1_inputs = []
for i in range(3):
    t = widgets.Text(placeholder=f'ユーザー名 {i+1}', layout=layout_full)
    t.add_class('widget-text')
    v1_inputs.append(t)

v1_btn = widgets.Button(description='誰が安全かチェック', button_style='success', layout=layout_btn)
v1_out = widgets.Output()

def on_v1_click(b):
    with v1_out:
        clear_output()
        raw_values = [w.value for w in v1_inputs]
        sanitized_values = [manager.sanitize(v) for v in raw_values]

        if not all(sanitized_values):
            display(widgets.HTML("<div style='color:#ff1744; text-align:center; font-weight:bold; padding:20px;'>⚠ 3名すべて入力してください</div>"))
            return

        safe, mines = manager.check_safety(raw_values)
        for i, w in enumerate(v1_inputs): w.value = sanitized_values[i]

        html = ""
        if safe:
            html += f"""
            <div class='hero-safe'>
                <div class='hero-title'>一番安全</div>
                <div class='hero-name'>✅ {' / '.join(safe)}</div>
            </div>
            """
        elif mines:
            min_level = mines[-1][1]
            better_options = [name for name, lv in mines if lv == min_level]
            better_names = ' / '.join(better_options)
            html += f"""
            <div class='hero-warning'>
                <div class='hero-title'>強いて言えば…</div>
                <div class='hero-name'>⚠ {better_names}</div>
                <div style='font-size:12px; margin-top:5px; opacity:0.8;'>地雷ですが一番マシです (Lv.{min_level})</div>
            </div>
            """

        if mines:
            html += "<div style='color:#aaa; font-size:12px; margin-bottom:5px;'>避けましょう</div>"
            for n, l in mines:
                html += f"""
                <div class='list-mine'>
                    <span>💀 <b>{n}</b></span>
                    <span style='background:#ff1744; color:white; padding:2px 10px; border-radius:12px; font-size:12px;'>Lv.{l}</span>
                </div>
                """
        display(widgets.HTML(html))

v1_btn.on_click(on_v1_click)
view1 = widgets.VBox([
    widgets.HTML("<div style='margin-bottom:15px; color:#aaa; font-weight:bold;'>安全性チェック (3名必須)</div>"),
    *v1_inputs, widgets.HTML("<div style='height:15px'></div>"), v1_btn, widgets.HTML("<hr style='border-color:#333'>"), v1_out
], layout=widgets.Layout(align_items='center'))

v2_input = widgets.Text(placeholder='ユーザー名', layout=layout_full)
v2_input.add_class('widget-text')
v2_btn = widgets.Button(description='追加 / 複数追加でレベルアップ', button_style='danger', layout=layout_btn)
v2_out = widgets.Output()

def on_v2_click(b):
    with v2_out:
        clear_output()
        suc, msg, cln, lv = manager.add_user(v2_input.value)
        if cln: v2_input.value = cln
        s = "border-color:#ff1744; color:#ff1744;" if "⚠" in msg else "border-color:#00e676; color:#00e676;"
        display(widgets.HTML(f"<div style='padding:15px; border:1px solid; border-radius:5px; {s} font-weight:bold; background:#1a1a1d;'>{msg}</div>"))

v2_btn.on_click(on_v2_click)
view2 = widgets.VBox([
    widgets.HTML("<div style='margin-bottom:15px; color:#aaa; font-weight:bold;'>新規登録 または 地雷報告</div>"),
    v2_input, widgets.HTML("<div style='height:15px'></div>"), v2_btn, widgets.HTML("<hr style='border-color:#333'>"), v2_out
], layout=widgets.Layout(align_items='center'))

v3_search = widgets.Text(placeholder='名前検索 (空欄で全件表示)', layout=layout_full)
v3_search.add_class('widget-text')
v3_btn = widgets.Button(description='データベース検索', button_style='warning', icon='search', layout=layout_btn)
v3_out = widgets.Output(layout=widgets.Layout(width='95%'))

def render_list(b=None):
    with v3_out:
        clear_output()
        matches = manager.search_in_list(v3_search.value)
        if not matches:
            display(widgets.HTML(f"<div style='color:#777; text-align:center; padding:20px;'>{'該当なし' if v3_search.value else 'データなし'}</div>"))
            return
        display(widgets.HTML(f"<div style='color:#888; margin-bottom:10px; border-bottom:1px solid #444;'>Hit: {len(matches)}件</div>"))
        items = []
        for n, lv in matches:
            row = f"<div style='display:flex; align-items:center;'><span style='color:#ff1744; font-weight:bold; width:50px;'>Lv.{lv}</span><span style='color:#eee;'>{n}</span></div>"
            d = widgets.Button(description="DEL", layout=widgets.Layout(width='50px', height='25px'))
            d.style.button_color = '#333'
            d.style.text_color = '#aaa'
            d.on_click(lambda b, name=n: (manager.delete_user(name), render_list()))
            items.append(widgets.HBox([widgets.HTML(row, layout=widgets.Layout(flex='1')), d], layout=widgets.Layout(border_bottom='1px solid #333', padding='8px 0')))
        display(widgets.VBox(items))

v3_btn.on_click(render_list)
view3 = widgets.VBox([
    widgets.HTML("<div style='margin-bottom:15px; color:#aaa; font-weight:bold;'>データベース検索・編集</div>"),
    v3_search, widgets.HTML("<div style='height:15px'></div>"), v3_btn, widgets.HTML("<hr style='border-color:#333'>"), v3_out
], layout=widgets.Layout(align_items='center'))

v4_wh = widgets.Text(placeholder='Webhook URL', layout=layout_full)
v4_wh.add_class('widget-text')
v4_btn = widgets.Button(description='discord共有', button_style='success', layout=layout_btn)
v4_out = widgets.Output()
v4_up = widgets.FileUpload(accept='.json', multiple=True, description='読み込み', layout=layout_full)
v4_up.add_class('widget-upload')
v4_io_out = widgets.Output()

def on_dl_click(b):
    manager.save_data()
    files.download(FILE_PATH)
dl_btn = widgets.Button(description='ダウンロード', icon='download', layout=layout_btn)
dl_btn.on_click(on_dl_click)

def on_up_change(change):
    if not change['new']: return
    with v4_io_out:
        clear_output()
        vals = change['owner'].value
        items = vals.items() if isinstance(vals, dict) else [(f.get('name', 'unknown'), f) for f in vals]
        for name, f_info in items:
            try:
                content_bytes = f_info['content'] if 'content' in f_info else f_info.get('content', b'{}')
                c = json.loads(content_bytes.decode('utf-8') if isinstance(content_bytes, bytes) else content_bytes)
                a, u, msg = manager.merge_data(c)
                display(widgets.HTML(f"<div style='color:#00e676; border:1px solid #00e676; padding:10px; margin-bottom:5px;'>✅ {name}<br>New: {a} / Upd: {u}</div>"))
            except Exception as e: print(f"Err {name}: {e}")
        try: v4_up.value.clear(); v4_up._counter = 0
        except: pass

v4_up.observe(on_up_change, names='value')

def on_v4_click(b):
    with v4_out:
        clear_output()
        success, msg = manager.send_to_webhook(v4_wh.value)
        color = '#00e676' if success else '#ff1744'
        display(widgets.HTML(f"<div style='color:{color}'>{msg}</div>"))

v4_btn.on_click(on_v4_click)

view4 = widgets.VBox([
    widgets.HTML("<div style='margin-bottom:5px; color:#aaa; font-weight:bold;'>Webhook共有</div>"),
    v4_wh, v4_btn, v4_out,
    widgets.HTML("<hr style='border-color:#333; margin:20px 0;'>"),
    widgets.HTML("<div style='margin-bottom:5px; color:#aaa; font-weight:bold;'>データ管理</div>"),
    dl_btn, widgets.HTML("<div style='height:10px'></div>"), v4_up, v4_io_out
], layout=widgets.Layout(align_items='center'))

tabs = [widgets.Button(description=t) for t in ["検索", "追加", "編集・削除", "データ関係"]]
views = [view1, view2, view3, view4]
content = widgets.VBox(layout=widgets.Layout(width='100%'))

def switch(idx):
    content.children = [views[idx]]
    for i, b in enumerate(tabs):
        b.remove_class('active-tab')
        b.add_class('custom-tab-btn')
        if i == idx: b.add_class('active-tab')

for i, b in enumerate(tabs):
    b.add_class('custom-tab-btn')
    b.on_click(lambda b, x=i: switch(x))

app = widgets.VBox([
    css,
    widgets.HTML(f"<div class='app-header'><div class='app-title'>地雷管理ﾁｬﾝ <span style='font-size:12px; color:#666; margin-left:10px;'>作成者:@yu_</span></div></div>"),
    widgets.HBox(tabs, layout=widgets.Layout(width='100%')),
    widgets.VBox([content], layout=widgets.Layout(padding='0'))
])
app.add_class('app-container')

switch(0)
display(app)